In [1]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import yfinance as yf
import ta
import math
from fredapi import Fred
import logging
import ecbdata
Fred_api="e886df7269c2c4e6209754d4ea0371d5"
# kostis key 16e61637be609fe4e702a331066b0ebf
# tsu key e886df7269c2c4e6209754d4ea0371d5

# List Of Tradeable Pairs And Indicators

In [2]:
# Initialize MetaTrader 5 connection
mt5.initialize()

# Updated list of currency pairs
pairs = [
    "EURUSD",  # Euro / US Dollar
    "EURCHF",  # Euro / Swiss Franc
    "EURJPY",  # Euro / Japanese Yen
    "USDCHF",  # US Dollar / Swiss Franc
    "CHFJPY",  # Swiss Franc / Japanese Yen
    
    "USDJPY"   # US Dollar / Japanese Yen
]

currencies = [
   "DX-Y.NYB", # Dollar Currency Index
    "^XDE",    # Euro Currency Index
    "^XDS",    # Chf Currency Index
    "^XDN"     # Yen Currency Index
]

# Function to get the latest Ask and Bid prices for a given pair
def get_latest_prices(symbol):
    # Get the latest tick data for the symbol
    tick = mt5.symbol_info_tick(symbol)
    if tick is None:
        print(f"Failed to get latest tick data for {symbol}")
        return None, None
    return tick.ask, tick.bid

# Function to get historical data for a given pair
def get_historical_data(symbol, timeframe=mt5.TIMEFRAME_M15, n_bars=64):
    # Fetch historical data
    rates = mt5.copy_rates_from_pos(symbol, timeframe, 0, n_bars)
    if rates is None or len(rates) == 0:
        print(f"Failed to get historical data for {symbol}")
        return None
    data = pd.DataFrame(rates)
    data['time'] = pd.to_datetime(data['time'], unit='s')
    return data

# Function to calculate EMA, RSI, and ATR for a given pair
def calculate_indicators(symbol):
    # Get historical data for the pair
    data = get_historical_data(symbol)
    if data is None:
        return None, None, None, None

    # Calculate EMA 64
    data['EMA_64'] = ta.trend.ema_indicator(data['close'], window=64)

    # Calculate RSI 16
    data['RSI_16'] = ta.momentum.rsi(data['close'], window=16)

    # Calculate ATR 16
    data['ATR_16'] = ta.volatility.average_true_range(data['high'], data['low'], data['close'], window=16)

    # Get the latest values of the indicators
    latest_price = data['close'].iloc[-1]
    latest_ema = data['EMA_64'].iloc[-1]
    latest_rsi = data['RSI_16'].iloc[-1]
    latest_atr = data['ATR_16'].iloc[-1]

    return latest_price, latest_ema, latest_rsi, latest_atr

# Macroeconomic Data

In [3]:
import pandas as pd
from fredapi import Fred
import logging
from ecbdata import ecbdata

# Initialize FRED with your API key
fred = Fred(api_key=Fred_api)  # Replace with your actual API key

# Define the countries and their respective indicators
countries = {
    'USA': {'gdp': 'GDPC1', 'unemp': 'UNRATE', 'interest': 'FEDFUNDS', 'inflation': 'CPIAUCSL'},
    'Europe': {'gdp': 'CLVMEURSCAB1GQEA19', 'unemp': 'LRUNTTTTQZA156S', 'interest': 'IR3TIB01EZQ156N', 'inflation': 'CPHPTT01EZQ659N'},
    'Switzerland': {'gdp': 'CLVMNACSAB1GQCH', 'unemp': 'LRUNTTTTCHQ156S', 'interest': 'IR3TIB01CHQ156N', 'inflation': 'CHECPIALLMINMEI'},
    'Japan': {'gdp': 'JPNRGDPEXP', 'unemp': 'LRUNTTTTJPQ156S', 'interest': 'IR3TIB01JPQ156N', 'inflation': 'JPNCPIALLMINMEI'}
}


# Initialize empty dictionaries for storing the economic data
growth_rates = {}
unemp_rates = {}
interest_rates = {}
inflation_rates = {}
fallbacks = {
    'Europe': {
        'inflation': 'ICP.M.U2.N.000000.4.ANR',  # Alternative Europe inflation series
        'unemp': 'LFSI.M.U2.N.UNEHRT.TOTAL0.15_74.T',  # Alternative Europe unemployment series
    }
}

def get_10_years_Edata(symbol, country, indicator):
    try:
        economic_data = fred.get_series(symbol)
        economic_data = economic_data[economic_data.index >= pd.Timestamp.now() - pd.DateOffset(years=10)]
        if economic_data.empty:
            raise ValueError(f"No data returned for {symbol}")
        logging.info(f"Successfully fetched {indicator} data for {country} using {symbol} ({len(economic_data)} points)")
        return economic_data
    except Exception as e:
        logging.error(f"Error fetching {indicator} data for {country}: {e}")
        return None
def calculate_growth_rate(gdp_series):
    """Calculate annualized quarterly GDP growth rate"""
    if gdp_series is None or len(gdp_series) < 2:
        return None
    gdp_now = gdp_series.iloc[-1]
    gdp_previous = gdp_series.iloc[-2]
    return ((gdp_now - gdp_previous) / gdp_previous) * 100 * 4

def get_latest_value(series):
    """Get the most recent value from a series"""
    if series is None or len(series) < 1:
        return None
    return series.iloc[-1]

def calculate_inflation_rate(cpi_series):
    """Calculate year-over-year inflation rate"""
    if cpi_series is None or len(cpi_series) < 13:  # Need 12 months + 1 for monthly data
        logging.warning(f"Insufficient data for inflation calculation: {len(cpi_series)} points")
        return None
    cpi_now = cpi_series.iloc[-1]
    cpi_year_ago = cpi_series.iloc[-13]  # Assumes monthly data
    return ((cpi_now - cpi_year_ago) / cpi_year_ago) * 100

# Update the economic data and store it in a DataFrame
def update_economic_data():
    global growth_rates, unemp_rates, interest_rates, inflation_rates
    for country, symbols in countries.items():
        # GDP Growth
        gdp_data = get_10_years_Edata(symbols['gdp'], country, 'gdp')
        if gdp_data is not None:
            growth_rates[country] = calculate_growth_rate(gdp_data)
        
        # Unemployment
        unemp_data = get_10_years_Edata(symbols['unemp'], country, 'unemployment')
        if unemp_data is not None:
            unemp_rates[country] = get_latest_value(unemp_data)
            
        
        # Interest Rates
        interest_data = get_10_years_Edata(symbols['interest'], country, 'interest')
        if interest_data is not None:
            interest_rates[country] = get_latest_value(interest_data)
        
        # Inflation
        inflation_data = get_10_years_Edata(symbols['inflation'], country, 'inflation')
        if inflation_data is not None:
            if country == 'Europe' and symbols['inflation'] == 'CPHPTT01EZQ659N':
                inflation_rates[country] = get_eur_inflation()
            else:
                inflation_rates[country] = calculate_inflation_rate(inflation_data)
          
def get_eur_inflation():
    economic_data = ecbdata.get_series('ICP.M.U2.N.000000.4.ANR', start='2024-01')
    latest = economic_data['OBS_VALUE'].iloc[-1]  # Fixed: Use 'economic_data'
    return latest

def get_eur_unemployment():
    economic_data = ecbdata.get_series('LFSI.M.U2.N.UNEHRT.TOTAL0.15_74.T', start='2024-01')
    latest = economic_data['OBS_VALUE'].iloc[-1]  # Fixed: Use 'economic_data'
    return latest

# Create a DataFrame to hold the economic data
def create_economic_dataframe():
    update_economic_data()
    
    # Combine the data into a DataFrame
    data = {
        'GDP Growth': growth_rates,
        'Unemployment': unemp_rates,
        'Interest Rate': interest_rates,
        'Inflation Rate': inflation_rates
    }
    
    df = pd.DataFrame(data)
    df.loc['Japan', "Inflation Rate"] = 4
    df.loc['Europe', 'Unemployment'] = get_eur_unemployment() 
    df.loc['Europe', 'Inflation Rate'] =get_eur_inflation()
    return df
df = create_economic_dataframe()

df


ERROR:root:Error fetching unemployment data for Europe: Bad Request.  The series does not exist.
ERROR:root:Error fetching inflation data for Europe: No data returned for CPHPTT01EZQ659N


,GDP Growth,Unemployment,Interest Rate,Inflation Rate
USA,2.232707,4.000000,4.330000,2.999413
Europe,0.203240,6.268223,2.996487,2.500000
Switzerland,1.536775,4.497941,0.767250,0.401528
Japan,1.233826,2.466667,0.334667,4.000000


# Currency Index Data

In [4]:
def get_currency_data(currency):
    # Fetch DXY historical data (last 10 days, 15m interval)
    dxy = yf.Ticker(currency)
    data = dxy.history(period="10d", interval="15m")
    
    # Recalculate RSI & EMA
    data["RSI_16"] = ta.momentum.rsi(data["Close"], window=64)
    data["EMA_64"] = ta.trend.ema_indicator(data["Close"], window=196)

    latest_price = data['Close'].iloc[-1]
    latest_ema = data['EMA_64'].iloc[-1]
    latest_rsi = data['RSI_16'].iloc[-1]
    return latest_price, latest_ema, latest_rsi

def get_currencies_table(currencies):
    dict = {}
    for i in currencies:
        name = i
        if i == "DX-Y.NYB":
            name = "USD"
        elif i == "^XDE" :
            name = "EUR"
        elif i== "^XDS" :
            name = "CHF"
        elif i== "^XDN" :
            name = "JPY"
        
        latest_price, latest_ema, latest_rsi = get_currency_data(i)
        dict.update({name: [latest_price, latest_ema, latest_rsi ]})
    data = pd.DataFrame.from_dict(dict, orient='index', columns=['Price', 'EMA', 'RSI'])
    return data

get_currencies_table(currencies)

,Price,EMA,RSI
USD,106.502998,106.697717,46.004517
EUR,104.608002,104.396280,51.620836
CHF,111.445396,110.795702,58.203743
JPY,67.052399,66.205350,64.359872


# Pip Value

In [5]:
def get_pip_value(symbol):
    dec = 0.0001
    if "JPY" in symbol:
        dec = 0.01
    latest_price, latest_ema, latest_rsi, latest_atr = calculate_indicators(symbol)
    pip_value = (dec * 100000) / latest_price
    return pip_value

# Position Size

In [6]:
def get_position_size(pair, stop_loss):
    account_info = mt5.account_info()
    balance = account_info.balance
    risk_amount = 0.005 * balance
    size = risk_amount / ( stop_loss * get_pip_value(pair))
    size = round(size, 2)
    return size

# Bias

In [7]:
# from imfdatapy.imf import IMF

# Keep your existing dictionaries as global variables (assumed to be defined elsewhere)
# growth_rates, unemp_rates, interest_rates, inflation_rates

def update_debt_to_gdp():
    # """
    # Fetch debt-to-GDP ratios for 2025 (latest projection) and 2024 from IMF WEO for EU, US, Japan, and Switzerland.
    # Returns a dictionary with 'latest' (2025) and 'previous' (2024) sub-dictionaries for comparison.
    # """
    # try:
    #     imf = IMF()
    #     # Fetch "General government gross debt" (% of GDP) for EA (Euro Area), US, JP, CH
    #     debt_data = imf.get_series("WEO", "GGXWDG_NGDP", country=["EA", "US", "JP", "CH"], period="A")
        
    #     # Explicitly target 2025 (latest projection) and 2024 (previous year)
    #     debt_to_gdp = {
    #         "latest": {
    #             "EU": debt_data["EA"].loc[2025]["value"],
    #             "US": debt_data["US"].loc[2025]["value"],
    #             "Japan": debt_data["JP"].loc[2025]["value"],
    #             "Switzerland": debt_data["CH"].loc[2025]["value"]
    #         },
    #         "previous": {
    #             "EU": debt_data["EA"].loc[2024]["value"],
    #             "US": debt_data["US"].loc[2024]["value"],
    #             "Japan": debt_data["JP"].loc[2024]["value"],
    #             "Switzerland": debt_data["CH"].loc[2024]["value"]
    #         }
    #     }
    #     print(f"Debt-to-GDP fetched: Latest (2025 projection) vs Previous (2024)")
    #     return debt_to_gdp
    # except Exception as e:
    #     print(f"Error fetching debt-to-GDP data: {e}")
    #     # Fallback values for 2025 (projections) and 2024 based on trends from prior data
    return {
        "latest": {"EU": 89.0, "US": 122.0, "Japan": 261.0, "Switzerland": 43.5},  # 2025 projections
        "previous": {"EU": 88.5, "US": 120.0, "Japan": 260.0, "Switzerland": 43.3}  # 2024 estimates
    }

# Initialize debt_to_gdp dynamically
debt_to_gdp = update_debt_to_gdp()

def compare_economies(country1, country2):
    """
    Compare the economic data between two countries using latest data and year-over-year debt trends.
    Returns a 'buy' signal for country1 or 'sell' for country2 based on macroeconomic performance.
    Uses 5 factors, with debt-to-GDP trend as a tiebreaker, and a 2.5 threshold to minimize neutral outcomes.
    """
    buy_factors = 0
    sell_factors = 0
    
    # Compare GDP Growth Rates (weight: 1.5)
    if growth_rates.get(country1, 0) > growth_rates.get(country2, 0):
        buy_factors += 1.5
    elif growth_rates.get(country1, 0) < growth_rates.get(country2, 0):
        sell_factors += 1.5
    
    # Compare Unemployment Rates (lower is better, weight: 1)
    if unemp_rates.get(country1, float('inf')) < unemp_rates.get(country2, float('inf')):
        buy_factors += 1
    elif unemp_rates.get(country1, float('inf')) > unemp_rates.get(country2, float('inf')):
        sell_factors += 1
    
    # Compare Interest Rates (higher is generally better, weight: 1)
    if interest_rates.get(country1, 0) > interest_rates.get(country2, 0):
        buy_factors += 1
    elif interest_rates.get(country1, 0) < interest_rates.get(country2, 0):
        sell_factors += 1
    
    # Compare Inflation Rates (lower is better, weight: 1.5)
    if inflation_rates.get(country1, float('inf')) < inflation_rates.get(country2, float('inf')):
        buy_factors += 1.5
    elif inflation_rates.get(country1, float('inf')) > inflation_rates.get(country2, float('inf')):
        sell_factors += 1.5
    
    # Compare Debt-to-GDP Trend (lower latest vs previous is better, weight: 0.5)
    debt_trend1 = debt_to_gdp["latest"].get(country1, float('inf')) - debt_to_gdp["previous"].get(country1, float('inf'))
    debt_trend2 = debt_to_gdp["latest"].get(country2, float('inf')) - debt_to_gdp["previous"].get(country2, float('inf'))
    if debt_trend1 < debt_trend2:  # Smaller increase or larger decrease favors country1
        buy_factors += 0.5
    elif debt_trend1 > debt_trend2:
        sell_factors += 0.5
    
    # Determine the final bias with a 2.5 threshold to minimize neutral outcomes
    if buy_factors > 2.5:
        return 'buy'
    elif sell_factors > 2.5:
        return 'sell'
    else:
        return 'neutral'

def bias_for_pairs(pairs):
    bias_results = {}
    
    # Update debt_to_gdp before running comparisons
    global debt_to_gdp
    debt_to_gdp = update_debt_to_gdp()
    
    # Loop through each pair and get the macroeconomic comparison
    for i in pairs:
        if "EUR" in i and "USD" in i:
            bias = compare_economies("EU", "US")  # Compare Eurozone with USA
        elif "EUR" in i and "CHF" in i:
            bias = compare_economies("EU", "Switzerland")  # Compare Eurozone with Switzerland
        elif "EUR" in i and "JPY" in i:
            bias = compare_economies("EU", "Japan")  # Compare Eurozone with Japan
        elif "USD" in i and "CHF" in i:
            bias = compare_economies("US", "Switzerland")  # Compare USA with Switzerland
        elif "CHF" in i and "JPY" in i:
            bias = compare_economies("Switzerland", "Japan")  # Compare Switzerland with Japan
        elif "USD" in i and "JPY" in i:
            bias = compare_economies("US", "Japan")  # Compare USA with Japan
        
        # Update the dictionary with the bias result
        bias_results[i] = bias
    
    return bias_results


# Sentiment 

In [8]:
def get_sentiment(pair):
    table = get_currencies_table(currencies)
    signal1 = "neutral"
    signal2 = "neutral"
    sentiment = "neutral"
    for index in table.index:
        if pair.startswith(index):
            currency1 = index
        elif pair.endswith(index):
            currency2 = index
    if table.loc[currency1, "Price"] > table.loc[currency1, "EMA"] and table.loc[currency1, "RSI"] > 50:
        signal1 = "buy"
    elif table.loc[currency1, "Price"] < table.loc[currency1, "EMA"] and table.loc[currency1, "RSI"] < 50:
        signal1 = "sell"
    if table.loc[currency2, "Price"] > table.loc[currency2, "EMA"] and table.loc[currency2, "RSI"] > 50:
        signal2 = "buy"
    elif table.loc[currency2, "Price"] < table.loc[currency2, "EMA"] and table.loc[currency2, "RSI"] < 50:
        signal2 = "sell"
    if signal1 == "buy" and signal2 == "sell":
        sentiment = "buy"
    elif signal1 == "sell" and signal2 == "buy":
        sentiment = "sell"
    return sentiment

for i in pairs:
    print(i, get_sentiment(i))
    print()

EURUSD buy

EURCHF neutral

EURJPY neutral

USDCHF sell

CHFJPY neutral

USDJPY sell



# Signals

In [9]:
def detect_rsi_divergence(symbol, timeframe=mt5.TIMEFRAME_M15, n_bars=64):
    # Fetch historical data
    data = get_historical_data(symbol, timeframe, n_bars)
    if data is None or len(data) < 20:  # Need enough data for meaningful analysis
        return "No Divergence"

    # Calculate RSI (using your existing 16-period RSI)
    data['RSI_16'] = ta.momentum.rsi(data['close'], window=16)

    # Identify peaks and troughs in price and RSI (using a 5-bar window to detect pivots)
    data['price_high'] = data['close'][(data['close'] > data['close'].shift(1)) & 
                                       (data['close'] > data['close'].shift(-1)) & 
                                       (data['close'] > data['close'].shift(2)) & 
                                       (data['close'] > data['close'].shift(-2))]
    data['price_low'] = data['close'][(data['close'] < data['close'].shift(1)) & 
                                      (data['close'] < data['close'].shift(-1)) & 
                                      (data['close'] < data['close'].shift(2)) & 
                                      (data['close'] < data['close'].shift(-2))]
    data['rsi_high'] = data['RSI_16'][(data['RSI_16'] > data['RSI_16'].shift(1)) & 
                                      (data['RSI_16'] > data['RSI_16'].shift(-1)) & 
                                      (data['RSI_16'] > data['RSI_16'].shift(2)) & 
                                      (data['RSI_16'] > data['RSI_16'].shift(-2))]
    data['rsi_low'] = data['RSI_16'][(data['RSI_16'] < data['RSI_16'].shift(1)) & 
                                     (data['RSI_16'] < data['RSI_16'].shift(-1)) & 
                                     (data['RSI_16'] < data['RSI_16'].shift(2)) & 
                                     (data['RSI_16'] < data['RSI_16'].shift(-2))]

    # Get the most recent two highs and lows
    price_highs = data['price_high'].dropna().tail(2)
    price_lows = data['price_low'].dropna().tail(2)
    rsi_highs = data['rsi_high'].dropna().tail(2)
    rsi_lows = data['rsi_low'].dropna().tail(2)

    # Check if we have enough pivots to compare
    if len(price_highs) < 2 or len(price_lows) < 2 or len(rsi_highs) < 2 or len(rsi_lows) < 2:
        return "No Divergence"

    # Extract values
    ph1, ph2 = price_highs.iloc[0], price_highs.iloc[1]
    pl1, pl2 = price_lows.iloc[0], price_lows.iloc[1]
    rh1, rh2 = rsi_highs.iloc[0], rsi_highs.iloc[1]
    rl1, rl2 = rsi_lows.iloc[0], rsi_lows.iloc[1]

    # Regular Divergence (Reversal)
    if pl2 < pl1 and rl2 > rl1 and rl2 < 40:  # Bullish: Price lower low, RSI higher low
        return "buy"
    elif ph2 > ph1 and rh2 < rh1 and rh2 > 60:  # Bearish: Price higher high, RSI lower high
        return "sell"

    # Hidden Divergence (Continuation)
    if pl2 > pl1 and rl2 < rl1 and rl2 < 50:  # Bullish: Price higher low, RSI lower low
        return "buy"
    elif ph2 < ph1 and rh2 > rh1 and rh2 > 50:  # Bearish: Price lower high, RSI higher high
        return "sell"

    return "neutral"


# Forex Pairs Data table

In [10]:
def get_data_table(pairs):
    dict = {}
    bias_results = bias_for_pairs(pairs)
    for i in pairs:
        ask, bid = get_latest_prices(i)
        latest_price, latest_ema, latest_rsi, latest_atr = calculate_indicators(i)
        dict.update({i: [ask,bid,latest_price, latest_ema, latest_rsi, latest_atr]})
    # Convert to DataFrame
    data = pd.DataFrame.from_dict(dict, orient='index', columns=['Ask', 'Bid', 'price', 'EMA', 'RSI', 'ATR'])
    
    data['Stop Loss']= data['ATR'] * 4
    data['Take Profit'] = data['Stop Loss'] * 2
    for i in pairs:
        dec = 10000
        if "JPY" in i:
            dec = 100
        stop_loss = round(data.loc[i, "Stop Loss"] * dec)
        take_profit = round(data.loc[i, "Take Profit"] * dec)
        data.loc[i,'Round Stop Loss'] = int(stop_loss)
        data.loc[i,'Round Take Profit'] = int(take_profit)
        data.loc[i,'Pip Value'] = get_pip_value(i)
        data.loc[i,'Position Size'] = get_position_size(i, stop_loss)
        data.loc[i, 'Bias'] = bias_results.get(i)
        data.loc[i,'Sentiment'] = get_sentiment(i)
        data.loc[i,'Signal'] = detect_rsi_divergence(i, timeframe=mt5.TIMEFRAME_M15, n_bars=64)
    return data
        
data = get_data_table(pairs)

data





,Ask,Bid,price,EMA,RSI,ATR,Stop Loss,Take Profit,Round Stop Loss,Round Take Profit,Pip Value,Position Size,Bias,Sentiment,Signal
EURUSD,1.04792,1.04789,1.04789,1.046927,65.709083,0.000717,0.002868,0.005736,29.0,57.0,9.542986,1.80,neutral,buy,neutral
EURCHF,0.94115,0.94100,0.94100,0.940164,63.211111,0.000695,0.002782,0.005564,28.0,56.0,10.626993,1.67,sell,neutral,neutral
EURJPY,156.43200,156.41000,156.41000,156.657109,54.677759,0.141936,0.567743,1.135486,57.0,114.0,6.393453,1.36,sell,neutral,neutral
USDCHF,0.89809,0.89796,0.89796,0.898024,52.572351,0.000634,0.002537,0.005074,25.0,51.0,11.136354,1.79,sell,sell,neutral
CHFJPY,166.23400,166.20600,166.20600,166.598054,44.711123,0.150666,0.602665,1.205330,60.0,121.0,6.016666,1.38,buy,neutral,buy
USDJPY,149.27100,149.26400,149.26400,149.632545,45.353128,0.108936,0.435743,0.871486,44.0,87.0,6.699539,1.69,sell,sell,neutral


# Decision Taking Order Code

In [11]:
import MetaTrader5 as mt5
import time
from datetime import datetime

# Signal history to track last signal per pair
signal_history = {pair: "neutral" for pair in pairs}

# Scoring function
def map_to_score(value, component):
    if component == 'bias':
        return 1 if value == 'buy' else -1 if value == 'sell' else 0
    elif component == 'sentiment':
        return 1 if value == 'buy' else -1 if value == 'sell' else 0
    elif component == 'signal':
        return 1 if value == 'buy' else -1 if value == 'sell' else 0

def calculate_score(bias, sentiment, signal):
    bias_score = map_to_score(bias, 'bias')
    sentiment_score = map_to_score(sentiment, 'sentiment')
    signal_score = map_to_score(signal, 'signal')
    return (0.40 * bias_score) + (0.35 * sentiment_score) + (0.25 * signal_score)

# Position size adjustment based on score strength
def adjust_position_size(base_size, score):
    if score == 0:
        return 0
    adjusted_size = base_size * (1 + abs(score))  # Scale with score strength
    return min(adjusted_size, base_size * 2)  # Cap at 2x

# Check if market is open for a symbol
def is_market_open(symbol):
    tick = mt5.symbol_info_tick(symbol)
    if tick and tick.time:
        last_tick_time = datetime.fromtimestamp(tick.time)
        current_time = datetime.now()
        return (current_time - last_tick_time).total_seconds() < 300  # 5-minute threshold
    return False

# Send order to MT5 with stop validation
def send_order(symbol, action, volume, stop_loss, take_profit):
    if not mt5.initialize():
        return False, "MT5 initialization failed"
    
    if not is_market_open(symbol):
        return False, f"Cannot trade {symbol}: Market is closed"
    
    symbol_info = mt5.symbol_info(symbol)
    if not symbol_info:
        return False, f"Failed to get symbol info for {symbol}"
    
    tick_size = symbol_info.point
    min_stop_distance = symbol_info.trade_stops_level * tick_size
    
    price = mt5.symbol_info_tick(symbol).ask if action in ['buy', 'strong buy'] else mt5.symbol_info_tick(symbol).bid
    sl_price = price - stop_loss if action in ['buy', 'strong buy'] else price + stop_loss
    tp_price = price + take_profit if action in ['buy', 'strong buy'] else price - take_profit
    
    if abs(price - sl_price) < min_stop_distance:
        stop_loss = min_stop_distance * 1.5
        sl_price = price - stop_loss if action in ['buy', 'strong buy'] else price + stop_loss
    if abs(price - tp_price) < min_stop_distance:
        take_profit = min_stop_distance * 3
        tp_price = price + take_profit if action in ['buy', 'strong buy'] else price - take_profit
    
    order_type = mt5.ORDER_TYPE_BUY if action in ['buy', 'strong buy'] else mt5.ORDER_TYPE_SELL
    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": symbol,
        "volume": round(volume, 2),
        "type": order_type,
        "price": price,
        "sl": sl_price,
        "tp": tp_price,
        "deviation": 10,
        "magic": 234000,
        "comment": "Automated Trade",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_IOC,
    }
    
    result = mt5.order_send(request)
    if result.retcode != mt5.TRADE_RETCODE_DONE:
        return False, f"Order failed for {symbol}: {result.comment}"
    return True, f"{symbol} {action} {volume} lots, Score: {score:.2f}"

# Main loop with 30-minute summary
if not mt5.initialize():
    print("MT5 initialization failed. Exiting.")
    exit()

last_summary_time = None
actions_taken = []

while True:
    try:
        current_time = datetime.now()
        data = get_data_table(pairs)
        market_open = False
    
        for pair in pairs:
            bias = data.loc[pair, 'Bias']
            sentiment = data.loc[pair, 'Sentiment']
            signal = data.loc[pair, 'Signal']
            base_size = data.loc[pair, 'Position Size']
            stop_loss = data.loc[pair, 'Round Stop Loss'] / (10000 if "JPY" not in pair else 100)
            take_profit = data.loc[pair, 'Round Take Profit'] / (10000 if "JPY" not in pair else 100)
            
            if is_market_open(pair):
                market_open = True
                last_signal = signal_history[pair]
                if signal != last_signal and signal != "neutral":
                    score = calculate_score(bias, sentiment, signal)
                    
                    if score <= -0.71:
                        action = 'strong sell'
                    elif score < 0:
                        action = 'sell'
                    elif score == 0:
                        action = None
                    elif score <= 0.71:
                        action = 'buy'
                    else:
                        action = 'strong buy'
                    
                    if action:
                        adjusted_size = adjust_position_size(base_size, score)
                        if adjusted_size > 0:
                            success, message = send_order(pair, action, adjusted_size, stop_loss, take_profit)
                            if success:
                                actions_taken.append(f"Trade executed: {message}")
                                signal_history[pair] = signal
                            else:
                                actions_taken.append(f"Failed: {message}")

        # Print summary every 30 minutes
        if last_summary_time is None or (current_time - last_summary_time).total_seconds() >= 1800:
            if not market_open:
                print(f"30-minute summary at {current_time}: Market closed - no trading possible")
            elif actions_taken:
                print(f"30-minute summary at {current_time}:")
                for action in actions_taken:
                    print(f"  {action}")
            else:
                print(f"30-minute summary at {current_time}: No actions taken")
            actions_taken = []  # Reset actions list
            last_summary_time = current_time
    
    except Exception as e:
        print(f"Error in cycle: {e}")
    
    time.sleep(60)  # Check every minute

30-minute summary at 2025-02-24 01:35:43.292517:
  Trade executed: CHFJPY buy 2.2769999999999997 lots, Score: 0.65


KeyboardInterrupt: 